# Multi-Head Latent Attention (MLA)

**Reference:** DeepSeek-V2 (2024) — *DeepSeek-V2: A Strong, Economical, and Efficient Mixture-of-Experts Language Model*

## Overview

Multi-Head Latent Attention compresses the KV cache into a low-rank **latent representation**, dramatically reducing memory during inference while maintaining expressive multi-head attention.

### Key Ideas

| Component | Description |
|---|---|
| **KV Compression** | Project hidden states to a small latent $c^{KV} \in \mathbb{R}^{d_c}$, then decompress to per-head K & V |
| **Query Compression** | Similarly compress queries to $c^Q \in \mathbb{R}^{d_c'}$ before decompressing per-head |
| **Decoupled RoPE** | Apply Rotary Position Embeddings on *separate* low-dimensional projections, keeping positional info independent |
| **KV Cache** | Only cache $c^{KV}$ and $k^{R}$ (RoPE keys) — *orders of magnitude* smaller than full KV |
| **Absorption** | During inference, absorb decompression matrices into Q/O projections so we *never* decompress the latent — pure cache-to-output |

### Mathematical Formulation

For input $h_t \in \mathbb{R}^d$:

$$c_t^{KV} = W^{DKV} h_t, \quad k_t^C = W^{UK} c_t^{KV}, \quad v_t = W^{UV} c_t^{KV}$$

$$c_t^Q = W^{DQ} h_t, \quad q_t^C = W^{UQ} c_t^Q$$

$$q_t^R = \text{RoPE}(W^{QR} c_t^Q), \quad k_t^R = \text{RoPE}(W^{KR} h_t)$$

$$q_t = [q_t^C;\; q_t^R], \quad k_t = [k_t^C;\; k_t^R]$$

In [14]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from typing import Optional, Dict

print(f"PyTorch {torch.__version__}")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

PyTorch 2.7.1+cpu
Device: cpu


## 1. Rotary Position Embedding (RoPE)

RoPE encodes position information by applying a rotation matrix to query and key vectors.
For a vector $x \in \mathbb{R}^d$, the rotation at position $t$ is:

$$\text{RoPE}(x, t)_i = \begin{cases} x_i \cos(t\theta_j) - x_{i+d/2} \sin(t\theta_j) & \text{if } i < d/2 \\ x_i \cos(t\theta_j) + x_{i-d/2} \sin(t\theta_j) & \text{if } i \geq d/2 \end{cases}$$

where $\theta_j = 10000^{-2j/d}$.

**In MLA, RoPE is "decoupled"**: it operates on a *separate* low-dimensional projection ($d_r$ dims per head) rather than on the full head dimension. This allows the content-based K/V to remain position-free inside the latent, making the absorption trick possible.

In [13]:
class RotaryPositionEmbedding(nn.Module):
    """Precomputes cos/sin tables for Rotary Position Embeddings."""

    def __init__(self, dim: int, max_seq_len: int = 4096, base: float = 10000.0):
        super().__init__()
        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
        self.register_buffer("inv_freq", inv_freq)  # [dim/2]
        self.max_seq_len = max_seq_len

    def forward(self, seq_len: int, device: torch.device):
        """Returns (cos, sin) each of shape [seq_len, dim]."""
        t = torch.arange(seq_len, device=device, dtype=self.inv_freq.dtype)
        freqs = torch.outer(t, self.inv_freq)          # [seq_len, dim/2]
        emb = torch.cat([freqs, freqs], dim=-1)         # [seq_len, dim]
        return emb.cos(), emb.sin()


def apply_rotary_emb(x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor) -> torch.Tensor:
    """
    Apply rotary embeddings to x.
    x: [..., dim]  (last dimension = dim)
    cos, sin: broadcastable to x's shape
    """
    d = x.shape[-1]
    x1, x2 = x[..., : d // 2], x[..., d // 2 :]
    rotated = torch.cat([-x2, x1], dim=-1)
    return x * cos + rotated * sin


# ── Quick sanity check ──
rope = RotaryPositionEmbedding(dim=32)
cos, sin = rope(seq_len=8, device="cpu")
print(f"RoPE cos shape: {cos.shape}  sin shape: {sin.shape}")
# Verify norm-preserving: ||RoPE(x)|| ≈ ||x||
x = torch.randn(2, 8, 32)
x_rot = apply_rotary_emb(x, cos.unsqueeze(0), sin.unsqueeze(0))
print(f"Norm before RoPE: {x.norm(dim=-1).mean():.4f}  after: {x_rot.norm(dim=-1).mean():.4f}")

RoPE cos shape: torch.Size([8, 32])  sin shape: torch.Size([8, 32])
Norm before RoPE: 5.4963  after: 5.4963


## 2. Compressed KV Cache

### Standard MHA vs MLA Cache

| | Standard MHA | MLA |
|---|---|---|
| **Cached entries** | $K \in \mathbb{R}^{T \times n_h d_h}$, $V \in \mathbb{R}^{T \times n_h d_h}$ | $c^{KV} \in \mathbb{R}^{T \times d_c}$, $k^R \in \mathbb{R}^{T \times d_r}$ |
| **Per-token bytes** (fp16) | $2 \times 2 \times n_h \times d_h$ | $2(d_c + d_r)$ |
| **Example** ($n_h{=}8, d_h{=}64, d_c{=}128, d_r{=}32$) | 2 048 B | 320 B (**6.4× smaller**) |

We only need to cache the **compressed latent** $c^{KV}$ and the **RoPE-applied key** $k^R$. During inference, full keys and values are *never* materialized — the absorption trick (Section 4) operates directly on the latent.

In [12]:
class CompressedKVCache:
    """
    KV cache for Multi-Head Latent Attention.

    Instead of caching full K ∈ [T, n_h·d_h] and V ∈ [T, n_h·d_h],
    we only store:
      • c_kv : compressed latent  [B, T, d_c]
      • k_rope: RoPE-applied keys [B, T, d_r]   (shared across heads)

    This gives a compression ratio of  2·n_h·d_h / (d_c + d_r).
    """

    def __init__(self):
        self.c_kv: Optional[torch.Tensor] = None    # [B, T_cached, d_kv_comp]
        self.k_rope: Optional[torch.Tensor] = None  # [B, T_cached, d_rope]

    @property
    def seq_len(self) -> int:
        """Number of tokens currently in cache."""
        return 0 if self.c_kv is None else self.c_kv.shape[1]

    def update(self, new_c_kv: torch.Tensor, new_k_rope: torch.Tensor):
        """Append new tokens to the cache."""
        if self.c_kv is None:
            self.c_kv = new_c_kv
            self.k_rope = new_k_rope
        else:
            self.c_kv = torch.cat([self.c_kv, new_c_kv], dim=1)
            self.k_rope = torch.cat([self.k_rope, new_k_rope], dim=1)

    def reset(self):
        """Clear the cache."""
        self.c_kv = None
        self.k_rope = None

    def memory_bytes(self) -> int:
        """Approximate memory usage in bytes."""
        if self.c_kv is None:
            return 0
        return self.c_kv.nelement() * self.c_kv.element_size() + \
               self.k_rope.nelement() * self.k_rope.element_size()


# ── Demonstrate cache size comparison ──
B, T, n_h, d_h, d_c, d_r = 1, 1024, 8, 64, 128, 32
standard_kv_bytes = 2 * B * T * n_h * d_h * 2  # K + V, fp16
mla_cache_bytes = B * T * (d_c + d_r) * 2        # c_kv + k_rope, fp16
print(f"Standard MHA KV cache : {standard_kv_bytes:>10,} bytes")
print(f"MLA compressed cache  : {mla_cache_bytes:>10,} bytes")
print(f"Compression ratio     : {standard_kv_bytes / mla_cache_bytes:.1f}×")

Standard MHA KV cache :  2,097,152 bytes
MLA compressed cache  :    327,680 bytes
Compression ratio     : 6.4×


## 3. Multi-Head Latent Attention Module

The module supports two execution paths:

1. **Training path** — standard forward: decompress latent → form full Q/K/V → attention → output projection.
2. **Inference path (absorbed)** — avoids decompressing the latent by absorbing projection matrices:

### Projection Matrices

```
Compression                    Decompression
─────────────                  ──────────────
W_dkv : d  → d_c  (KV down)   W_uk : d_c  → n_h·d_h  (key up)
W_dq  : d  → d_c' (Q down)    W_uv : d_c  → n_h·d_h  (val up)
                               W_uq : d_c' → n_h·d_h  (query up)

RoPE projections               Output
────────────────               ──────
W_qr  : d_c' → n_h·d_r        W_o  : n_h·d_h → d
W_kr  : d    → d_r (shared)
```

In [25]:
class MultiHeadLatentAttention(nn.Module):
    """
    Multi-Head Latent Attention

    Supports:
      • Training forward (standard decompress → attend → project)
      • Inference with compressed KV cache + weight absorption
    """

    def __init__(
        self,
        d_model: int,
        n_heads: int,
        d_head: int,
        d_kv_comp: int,
        d_q_comp: int,
        d_rope: int,
        max_seq_len: int = 4096,
        use_absorb: bool = False,
    ):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_head = d_head
        self.d_kv_comp = d_kv_comp
        self.d_q_comp = d_q_comp
        self.d_rope = d_rope
        self.use_absorb = use_absorb  # default inference mode for this instance

        # ── KV compression / decompression ──
        self.W_dkv = nn.Linear(d_model, d_kv_comp, bias=False)          # down-project
        self.W_uk = nn.Linear(d_kv_comp, n_heads * d_head, bias=False)  # key up
        self.W_uv = nn.Linear(d_kv_comp, n_heads * d_head, bias=False)  # value up

        # ── Query compression / decompression ──
        self.W_dq = nn.Linear(d_model, d_q_comp, bias=False)            # down-project
        self.W_uq = nn.Linear(d_q_comp, n_heads * d_head, bias=False)   # query up

        # ── Decoupled RoPE projections ──
        self.W_qr = nn.Linear(d_q_comp, n_heads * d_rope, bias=False)   # per-head rope Q
        self.W_kr = nn.Linear(d_model, d_rope, bias=False)              # shared rope K

        # ── Output projection ──
        self.W_o = nn.Linear(n_heads * d_head, d_model, bias=False)

        # ── RoPE ──
        self.rope = RotaryPositionEmbedding(d_rope, max_seq_len)

        self.scale = math.sqrt(d_head + d_rope)

        # ── Absorbed weight cache (None = not yet computed) ──
        # Populated by precompute_absorbed_weights() before inference.
        # Set to None whenever weights may have changed (e.g. after optimizer step).
        self._W_qk: Optional[torch.Tensor] = None  # [n_h, d_q_comp, d_kv_comp]
        self._W_vo: Optional[torch.Tensor] = None  # [n_h, d_kv_comp, d_model]

    # ─────────────────────────────────────────────
    #  Absorbed weight pre-computation
    # ─────────────────────────────────────────────
    def precompute_absorbed_weights(self):
        """
        Fuse decompression matrices into single absorbed weights.
        Call once after training is done (or after loading a checkpoint)
        before running inference with use_absorb=True.

        QK absorption:  W_qk_h = W_uq_h^T @ W_uk_h   [d_q_comp, d_kv_comp]
        VO absorption:  W_vo_h = W_uv_h^T @ W_o_h^T   [d_kv_comp, d_model]
        """
        W_uq = self.W_uq.weight.view(self.n_heads, self.d_head, self.d_q_comp)
        W_uk = self.W_uk.weight.view(self.n_heads, self.d_head, self.d_kv_comp)
        self._W_qk = torch.einsum("hdq,hdc->hqc", W_uq, W_uk).detach()
        # [n_h, d_q_comp, d_kv_comp]

        W_uv = self.W_uv.weight.view(self.n_heads, self.d_head, self.d_kv_comp)
        W_o  = self.W_o.weight.view(self.d_model, self.n_heads, self.d_head)
        self._W_vo = torch.einsum("hdc,mhd->hcm", W_uv, W_o).detach()
        # [n_h, d_kv_comp, d_model]

        print(f"Absorbed weights pre-computed  "
              f"W_qk: {list(self._W_qk.shape)}  W_vo: {list(self._W_vo.shape)}")

    def invalidate_absorbed_weights(self):
        """Call after any optimizer step so stale fused weights are not used."""
        self._W_qk = None
        self._W_vo = None

    # ─────────────────────────────────────────────
    #  Standard Training Forward
    # ─────────────────────────────────────────────
    def _forward_train(
        self,
        x: torch.Tensor,
        mask: Optional[torch.Tensor] = None,
        kv_cache: Optional[CompressedKVCache] = None,
    ) -> torch.Tensor:
        B, S, D = x.shape

        # ── Compress ──
        c_kv = self.W_dkv(x)   # [B, S, d_kv_comp]
        c_q = self.W_dq(x)     # [B, S, d_q_comp]

        # ── RoPE keys (shared across heads) ──
        k_rope_new = self.W_kr(x)  # [B, S, d_rope]

        # Position offset for RoPE
        pos_offset = kv_cache.seq_len if kv_cache is not None else 0
        cos, sin = self.rope(S + pos_offset, x.device)
        cos_s = cos[pos_offset : pos_offset + S].unsqueeze(0)  # [1, S, d_rope]
        sin_s = sin[pos_offset : pos_offset + S].unsqueeze(0)

        k_rope_new = apply_rotary_emb(k_rope_new, cos_s, sin_s)  # [B, S, d_rope]

        # ── Update KV cache ──
        if kv_cache is not None:
            kv_cache.update(c_kv, k_rope_new)
            c_kv = kv_cache.c_kv        # [B, T, d_kv_comp]
            k_rope = kv_cache.k_rope    # [B, T, d_rope]
        else:
            k_rope = k_rope_new

        T = c_kv.shape[1]  # total sequence length with cache

        # ── Decompress Q, K, V (standard path) ──
        q_content = self.W_uq(c_q).view(B, S, self.n_heads, self.d_head).transpose(1, 2)
        #   [B, n_h, S, d_h]
        k_content = self.W_uk(c_kv).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        #   [B, n_h, T, d_h]
        v = self.W_uv(c_kv).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        #   [B, n_h, T, d_h]

        # ── RoPE queries ──
        q_rope = self.W_qr(c_q).view(B, S, self.n_heads, self.d_rope).transpose(1, 2)
        cos_q = cos_s.unsqueeze(1)  # [1, 1, S, d_rope]
        sin_q = sin_s.unsqueeze(1)
        q_rope = apply_rotary_emb(q_rope, cos_q, sin_q)  # [B, n_h, S, d_rope]

        # ── Expand shared k_rope for all heads ──
        k_rope_exp = k_rope.unsqueeze(1).expand(-1, self.n_heads, -1, -1)
        #   [B, n_h, T, d_rope]

        # ── Concatenate content + RoPE → full Q, K ──
        q = torch.cat([q_content, q_rope], dim=-1)       # [B, n_h, S, d_h+d_r]
        k = torch.cat([k_content, k_rope_exp], dim=-1)   # [B, n_h, T, d_h+d_r]

        # ── Attention ──
        scores = torch.matmul(q, k.transpose(-2, -1)) / self.scale  # [B, n_h, S, T]
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float("-inf"))
        attn = F.softmax(scores, dim=-1)

        o = torch.matmul(attn, v)  # [B, n_h, S, d_h]
        o = o.transpose(1, 2).contiguous().view(B, S, self.n_heads * self.d_head)
        return self.W_o(o)         # [B, S, d_model]

    # ─────────────────────────────────────────────
    #  Absorbed Inference Forward
    # ─────────────────────────────────────────────
    def _forward_absorbed(
        self,
        x: torch.Tensor,
        kv_cache: CompressedKVCache,
        mask: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        """
        Inference path that NEVER decompresses c_kv to full K/V.

        Absorption identities
        ─────────────────────
        Score (content part, per head h):
          q_h^C · (k_h^C)ᵀ = c^Q · W_qk_h · c^{KV}ᵀ   (W_qk = W_uq^T W_uk)

        Value + output (per head h):
          o_h · W_o_hᵀ = attn_h · c^{KV} · W_vo_h       (W_vo = W_uv^T W_o^T)

        Both W_qk and W_vo are pre-computed once via precompute_absorbed_weights().
        """
        assert self._W_qk is not None and self._W_vo is not None, (
            "Call mla.precompute_absorbed_weights() before absorbed inference."
        )

        B, S, D = x.shape

        # ── Compress ──
        c_kv_new = self.W_dkv(x)  # [B, S, d_kv_comp]
        c_q      = self.W_dq(x)   # [B, S, d_q_comp]

        # ── RoPE keys ──
        k_rope_new = self.W_kr(x)  # [B, S, d_rope]
        pos_offset = kv_cache.seq_len
        cos, sin = self.rope(S + pos_offset, x.device)
        cos_s = cos[pos_offset : pos_offset + S].unsqueeze(0)
        sin_s = sin[pos_offset : pos_offset + S].unsqueeze(0)
        k_rope_new = apply_rotary_emb(k_rope_new, cos_s, sin_s)

        # ── Update cache ──
        kv_cache.update(c_kv_new, k_rope_new)
        c_kv   = kv_cache.c_kv      # [B, T, d_kv_comp]
        k_rope = kv_cache.k_rope    # [B, T, d_rope]

        # ── RoPE queries ──
        q_rope = self.W_qr(c_q).view(B, S, self.n_heads, self.d_rope).transpose(1, 2)
        q_rope = apply_rotary_emb(q_rope, cos_s.unsqueeze(1), sin_s.unsqueeze(1))
        #   [B, n_h, S, d_rope]

        # ──────────────────────────────────────
        #  Content scores using pre-computed W_qk
        #  c_q @ W_qk_h @ c_kv^T  per head
        # ──────────────────────────────────────
        qk_latent = torch.einsum("bsq,hqc->bhsc", c_q, self._W_qk)
        #   [B, n_h, S, d_kv_comp]
        scores_content = torch.einsum("bhsc,btc->bhst", qk_latent, c_kv)
        #   [B, n_h, S, T]

        # ── RoPE scores ──
        k_rope_exp = k_rope.unsqueeze(1).expand(-1, self.n_heads, -1, -1)
        scores_rope = torch.matmul(q_rope, k_rope_exp.transpose(-2, -1))
        #   [B, n_h, S, T]

        scores = (scores_content + scores_rope) / self.scale
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float("-inf"))
        attn = F.softmax(scores, dim=-1)  # [B, n_h, S, T]

        # ──────────────────────────────────────
        #  Output using pre-computed W_vo
        #  attn @ c_kv @ W_vo_h  →  [B, S, d_model]
        # ──────────────────────────────────────
        c_kv_exp = c_kv.unsqueeze(1).expand(-1, self.n_heads, -1, -1)
        attn_c = torch.matmul(attn, c_kv_exp)                          # [B, n_h, S, d_kv_comp]
        output = torch.einsum("bhsc,hcm->bsm", attn_c, self._W_vo)    # [B, S, d_model]

        return output

    # ─────────────────────────────────────────────
    #  Unified forward
    # ─────────────────────────────────────────────
    def forward(
        self,
        x: torch.Tensor,
        mask: Optional[torch.Tensor] = None,
        kv_cache: Optional[CompressedKVCache] = None,
        use_absorb: Optional[bool] = None,
    ) -> torch.Tensor:
        """
        Args:
            x:          [B, S, d_model]
            mask:       [1, 1, S, T] broadcastable attention mask (1=attend, 0=mask)
            kv_cache:   CompressedKVCache instance (for incremental decoding)
            use_absorb: override the instance-level self.use_absorb for this call.
                        If None, falls back to self.use_absorb set in __init__.
                        Absorbed path requires precompute_absorbed_weights() first.

        Returns:
            output:     [B, S, d_model]
        """
        # Resolve which mode to use: per-call override > instance default
        absorb = self.use_absorb if use_absorb is None else use_absorb

        if absorb and not self.training:
            assert kv_cache is not None, "Absorbed path requires a KV cache"
            return self._forward_absorbed(x, kv_cache, mask)
        else:
            return self._forward_train(x, mask, kv_cache)


## 4. Weight Absorption — The Key Inference Trick

During autoregressive inference, we want to avoid decompressing $c^{KV}$ back to full-size $K$ and $V$ (which would negate the cache savings). The **absorption** trick fuses the decompression matrices into the query/output projections.

### Score Absorption (QK)

The content-based attention score for head $h$ is:

$$q_h^C \cdot (k_h^C)^\top = \underbrace{(c^Q W_{uq,h}^\top)}_{q_h^C} \cdot \underbrace{(c^{KV} W_{uk,h}^\top)^\top}_{(k_h^C)^\top}$$

$$= c^Q \cdot \underbrace{W_{uq,h}^\top W_{uk,h}}_{W_{qk,h}} \cdot (c^{KV})^\top$$

We pre-compute $W_{qk,h} \in \mathbb{R}^{d_c' \times d_c}$ **once**, then compute scores directly from the compressed representations.

### Value-Output Absorption (VO)

The output contribution from head $h$ is:

$$o_h \cdot W_{o,h}^\top = (\text{attn}_h \cdot c^{KV} \cdot W_{uv,h}^\top) \cdot W_{o,h}^\top = \text{attn}_h \cdot c^{KV} \cdot \underbrace{W_{uv,h}^\top W_{o,h}^\top}_{W_{vo,h}}$$

We pre-compute $W_{vo,h} \in \mathbb{R}^{d_c \times d}$ **once**, giving the final output as:

$$\text{output} = \sum_{h=1}^{n_h} \text{attn}_h \cdot c^{KV} \cdot W_{vo,h}$$


## 5. Demo & Verification

We verify that both execution paths produce **identical outputs** (within floating-point tolerance), confirming the absorption identities are implemented correctly.

In [22]:
# ── Hyperparameters ──
d_model   = 512   # model hidden dimension
n_heads   = 8     # number of attention heads
d_head    = 64    # per-head dimension
d_kv_comp = 128   # KV latent compression dimension
d_q_comp  = 192   # Query compression dimension
d_rope    = 32    # RoPE dimension per head
max_seq   = 4096

# ── Instantiate ──
mla = MultiHeadLatentAttention(
    d_model=d_model,
    n_heads=n_heads,
    d_head=d_head,
    d_kv_comp=d_kv_comp,
    d_q_comp=d_q_comp,
    d_rope=d_rope,
    max_seq_len=max_seq,
    use_absorb=True,   # use absorbed path by default at inference
).to(device)

total_params = sum(p.numel() for p in mla.parameters())
print(f"MLA parameters: {total_params:,}")
print(f"  W_dkv: {d_model}→{d_kv_comp}    W_uk: {d_kv_comp}→{n_heads*d_head}    W_uv: {d_kv_comp}→{n_heads*d_head}")
print(f"  W_dq:  {d_model}→{d_q_comp}    W_uq: {d_q_comp}→{n_heads*d_head}")
print(f"  W_qr:  {d_q_comp}→{n_heads*d_rope}    W_kr: {d_model}→{d_rope}")
print(f"  W_o:   {n_heads*d_head}→{d_model}")

MLA parameters: 720,896
  W_dkv: 512→128    W_uk: 128→512    W_uv: 128→512
  W_dq:  512→192    W_uq: 192→512
  W_qr:  192→256    W_kr: 512→32
  W_o:   512→512


In [23]:
# ══════════════════════════════════════════════
#  5a. Training forward pass (standard path)
# ══════════════════════════════════════════════
torch.manual_seed(42)
B, S = 2, 16  # batch size, sequence length
x = torch.randn(B, S, d_model, device=device)

# Causal mask: each position can only attend to itself and previous positions
causal_mask = torch.tril(torch.ones(S, S, device=device)).unsqueeze(0).unsqueeze(0)
#  [1, 1, S, S]

mla.train()
out_train = mla(x, mask=causal_mask)
print(f"Training output shape: {out_train.shape}")   # [B, S, d_model]
print(f"Output norm (mean):    {out_train.norm(dim=-1).mean():.4f}")

Training output shape: torch.Size([2, 16, 512])
Output norm (mean):    2.0550


In [24]:
# ══════════════════════════════════════════════
#  5b. Verify: Standard path == Absorbed path
#      (same full-sequence forward, no incremental cache)
# ══════════════════════════════════════════════
mla.eval()
mla.precompute_absorbed_weights()  # fuse W_qk and W_vo once

with torch.no_grad():
    # Standard path (override instance default with use_absorb=False)
    cache_std = CompressedKVCache()
    out_standard = mla(x, mask=causal_mask, kv_cache=cache_std, use_absorb=False)

    # Absorbed path  (uses instance default: self.use_absorb=True)
    cache_abs = CompressedKVCache()
    out_absorbed = mla(x, mask=causal_mask, kv_cache=cache_abs)

diff = (out_standard - out_absorbed).abs()
print(f"Standard output shape : {out_standard.shape}")
print(f"Absorbed output shape : {out_absorbed.shape}")
print(f"Max absolute diff     : {diff.max().item():.2e}")
print(f"Mean absolute diff    : {diff.mean().item():.2e}")
print(f"Outputs match         : {torch.allclose(out_standard, out_absorbed, atol=1e-4)} ✓")

Absorbed weights pre-computed  W_qk: [8, 192, 128]  W_vo: [8, 128, 512]
Standard output shape : torch.Size([2, 16, 512])
Absorbed output shape : torch.Size([2, 16, 512])
Max absolute diff     : 4.02e-07
Mean absolute diff    : 3.55e-08
Outputs match         : True ✓


In [ ]:
# ══════════════════════════════════════════════
#  5c. Autoregressive inference with KV cache
#      (token-by-token generation simulation)
# ══════════════════════════════════════════════
mla.eval()
mla.precompute_absorbed_weights()  # fuse W_qk and W_vo once
torch.manual_seed(42)

prompt_len = 8
gen_len = 8
x_full = torch.randn(1, prompt_len + gen_len, d_model, device=device)

# ── Step 1: Prefill — process the prompt in one shot ──
cache = CompressedKVCache()
prompt = x_full[:, :prompt_len, :]
with torch.no_grad():
    out_prefill = mla(prompt, kv_cache=cache)  # uses self.use_absorb=True
print(f"After prefill: cache has {cache.seq_len} tokens")

# ── Step 2: Decode — generate one token at a time ──
outputs = [out_prefill]
for t in range(gen_len):
    token = x_full[:, prompt_len + t : prompt_len + t + 1, :]  # [1, 1, d]
    with torch.no_grad():
        out_t = mla(token, kv_cache=cache)  # uses self.use_absorb=True
    outputs.append(out_t)
    if t < 3 or t == gen_len - 1:
        print(f"  Step {t}: cache = {cache.seq_len} tokens, "
              f"cache memory = {cache.memory_bytes():,} bytes")

all_outputs = torch.cat(outputs, dim=1)
print(f"\nFinal output shape: {all_outputs.shape}")
print(f"Total cache memory:  {cache.memory_bytes():,} bytes")

Absorbed weights pre-computed  W_qk: [8, 192, 128]  W_vo: [8, 128, 512]
After prefill: cache has 8 tokens
  Step 0: cache = 9 tokens, cache memory = 5,760 bytes
  Step 1: cache = 10 tokens, cache memory = 6,400 bytes
  Step 2: cache = 11 tokens, cache memory = 7,040 bytes
  Step 7: cache = 16 tokens, cache memory = 10,240 bytes

Final output shape: torch.Size([1, 16, 512])
Total cache memory:  10,240 bytes


In [ ]:
# ══════════════════════════════════════════════
#  5d. Verify: Incremental decoding == Full-sequence forward
#      (ensures cached computation is numerically consistent)
# ══════════════════════════════════════════════
mla.eval()
mla.precompute_absorbed_weights()  # fuse W_qk and W_vo once
torch.manual_seed(42)

seq_len = 12
x_test = torch.randn(1, seq_len, d_model, device=device)

# Causal mask for full-sequence forward
causal = torch.tril(torch.ones(seq_len, seq_len, device=device)).unsqueeze(0).unsqueeze(0)

with torch.no_grad():
    # Full-sequence forward with causal mask (override to standard path)
    cache_full = CompressedKVCache()
    out_full = mla(x_test, mask=causal, kv_cache=cache_full, use_absorb=False)

    # Incremental: prefill first half, then decode token by token (absorbed)
    cache_inc = CompressedKVCache()
    half = seq_len // 2

    # Prefill mask: causal for first half
    prefill_mask = torch.tril(torch.ones(half, half, device=device)).unsqueeze(0).unsqueeze(0)
    out_half = mla(x_test[:, :half, :], mask=prefill_mask, kv_cache=cache_inc)  # uses self.use_absorb=True

    inc_parts = [out_half]
    for t in range(half, seq_len):
        tok = x_test[:, t : t + 1, :]
        # Single-token attending to all cached + self → mask is all ones [1,1,1,T]
        T_so_far = cache_inc.seq_len + 1  # after update inside forward
        out_t = mla(tok, kv_cache=cache_inc, use_absorb=True)
        inc_parts.append(out_t)
    out_inc = torch.cat(inc_parts, dim=1)

diff = (out_full - out_inc).abs()
print(f"Full-sequence shape    : {out_full.shape}")
print(f"Incremental shape      : {out_inc.shape}")
print(f"Max absolute diff      : {diff.max().item():.2e}")
print(f"Mean absolute diff     : {diff.mean().item():.2e}")
print(f"Outputs match          : {torch.allclose(out_full, out_inc, atol=1e-4)} ✓")

Absorbed weights pre-computed  W_qk: [8, 192, 128]  W_vo: [8, 128, 512]
Full-sequence shape    : torch.Size([1, 12, 512])
Incremental shape      : torch.Size([1, 12, 512])
Max absolute diff      : 3.58e-07
Mean absolute diff     : 3.83e-08
Outputs match          : True ✓


## Summary

| Feature | Implementation |
|---|---|
| **KV Cache** | `CompressedKVCache` stores only $c^{KV} \in \mathbb{R}^{T \times d_c}$ and $k^R \in \mathbb{R}^{T \times d_r}$ — **6.4× smaller** than standard MHA |
| **RoPE** | Decoupled via separate projections $W^{QR}, W^{KR}$; applied to low-dim space independent of content K/V |
| **Absorption** | `_forward_absorbed()` fuses $W_{uq} / W_{uk}$ into $W_{qk}$ and $W_{uv} / W_o$ into $W_{vo}$, never decompressing the latent |
| **Verification** | Standard and absorbed paths produce numerically identical outputs; incremental decoding matches full-sequence forward |